# 01 — Data Ingestion (ETL)
## Mutual Fund Analytics | Day 1
---
**Objective:** Load all 10 CSV datasets, fetch live NAV from mfapi.in, explore fund master data, and validate AMFI codes.

## ⚙️ Setup & Dependencies

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import json
from pathlib import Path

# Go to project root if running from notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(f"📁 Working directory: {os.getcwd()}")

raw_path = Path("data/raw/")
print("✅ Libraries loaded successfully")


## 📂 Step 1 — Load All 10 CSV Datasets

In [ ]:
csv_files = {
    "fund_master":          "01_fund_master.csv",
    "nav_history":          "02_nav_history.csv",
    "aum_by_fund_house":    "03_aum_by_fund_house.csv",
    "monthly_sip_inflows":  "04_monthly_sip_inflows.csv",
    "category_inflows":     "05_category_inflows.csv",
    "industry_folio_count": "06_industry_folio_count.csv",
    "scheme_performance":   "07_scheme_performance.csv",
    "investor_transactions":"08_investor_transactions.csv",
    "portfolio_holdings":   "09_portfolio_holdings.csv",
    "benchmark_indices":    "10_benchmark_indices.csv",
}

dataframes = {}
print("=" * 60)
print("  LOADING ALL 10 CSV FILES")
print("=" * 60)

for name, filename in csv_files.items():
    df = pd.read_csv(raw_path / filename)
    dataframes[name] = df
    print(f"\n📄 {filename}")
    print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"   Columns : {list(df.columns)}")
    print(f"   Dtypes  :\n{df.dtypes.to_string()}")
    print(f"   Head(3) :\n{df.head(3).to_string()}")
    print(f"   Nulls   :\n{df.isnull().sum().to_string()}")
    print("-" * 60)

print(f"\n✅ Total files loaded: {len(dataframes)}")


## 🔍 Step 2 — Explore Fund Master Dataset

In [ ]:
fm = dataframes["fund_master"]

print("=" * 60)
print("  FUND MASTER — UNIQUE VALUES")
print("=" * 60)

print(f"\nFund Houses ({fm['fund_house'].nunique()}):")
for h in fm['fund_house'].unique():
    print(f"  - {h}")

print(f"\nCategories      : {list(fm['category'].unique())}")
print(f"Sub-Categories  : {list(fm['sub_category'].unique())}")
print(f"Risk Categories : {list(fm['risk_category'].unique())}")
print(f"SEBI Codes      : {list(fm['sebi_category_code'].unique())}")
print(f"\nAMFI Code structure (sample):")
print(fm[['amfi_code','scheme_name','fund_house']].head(5).to_string(index=False))


## 🌐 Step 3 — Fetch Live NAV from mfapi.in

In [ ]:
schemes = {
    "HDFC_Top100_Direct": 125497,
    "SBI_Bluechip":        119551,
    "ICICI_Bluechip":      120503,
    "Nippon_LargeCap":     118632,
    "Axis_Bluechip":       119092,
    "Kotak_Bluechip":      120841,
}

print("=" * 60)
print("  FETCHING LIVE NAV FROM mfapi.in")
print("=" * 60)

for name, code in schemes.items():
    try:
        url = f"https://api.mfapi.in/mf/{code}"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data["data"])
            df["scheme_name"] = name
            df["scheme_code"] = code
            output_path = raw_path / f"nav_{name}.csv"
            df.to_csv(output_path, index=False)
            print(f"  ✅ {name:30s} → {len(df):5,} records | Latest NAV: ₹{df.iloc[0]['nav']}")
        else:
            print(f"  ❌ {name} — HTTP {response.status_code}")
    except Exception as e:
        print(f"  ⚠️  {name} — {e}")

print("\n✅ All NAV data fetched and saved to data/raw/")


## ✅ Step 4 — Validate AMFI Codes

In [ ]:
fund_master = dataframes["fund_master"]
nav_history  = dataframes["nav_history"]

master_codes  = set(fund_master["amfi_code"])
history_codes = set(nav_history["amfi_code"])

missing = master_codes - history_codes
extra   = history_codes - master_codes

print("=" * 60)
print("  AMFI CODE VALIDATION")
print("=" * 60)
print(f"\nTotal codes in fund_master  : {len(master_codes)}")
print(f"Total codes in nav_history  : {len(history_codes)}")
print(f"Missing from nav_history    : {missing if missing else 'None ✅'}")
print(f"Extra in nav_history        : {extra if extra else 'None ✅'}")


## 📊 Step 5 — Data Quality Summary

In [ ]:
print("=" * 60)
print("  DATA QUALITY SUMMARY")
print("=" * 60)

for name, df in dataframes.items():
    total_nulls = df.isnull().sum().sum()
    status = "✅" if total_nulls == 0 else f"⚠️  {total_nulls} nulls"
    print(f"  {name:30s}: {df.shape[0]:6,} rows | {df.shape[1]:2} cols | Nulls: {status}")
    if total_nulls > 0:
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        for col, count in null_cols.items():
            print(f"      └── {col}: {count} nulls (expected for first-year YoY metrics)")

print("\n✅ Data ingestion complete — ready for Day 2 cleaning")


---
*Day 1: Data Ingestion complete | Bluestock MF Analytics Internship*